In [1]:
# =============================================================================
# PREDICCIÓN DE CLASIFICADOS A SEMIFINALES (CUARTOS DE FINAL)
# CON SCORE POR PARTIDO (Monte Carlo)
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from collections import defaultdict
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

print("🏆 PREDICCIÓN DE CLASIFICADOS A SEMIFINALES (CUARTOS DE FINAL)")
print("="*80 + "\n")

# ============================================================
# 1. CARGA Y LIMPIEZA DE DATOS
# ============================================================

base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

cruces_cuartos = pd.read_csv(f"{base_path}/Cruces_cuartos.csv", sep=',')
datos_historicos = pd.read_csv(f"{base_path}/datos_historicos.csv")
datos_mundial = pd.read_csv(f"{base_path}/datos_mundial.csv")
ranking_fifa = pd.read_csv(f"{base_path}/ranking_fifa.csv")
transfermarkt = pd.read_csv(f"{base_path}/transfermarkt.csv")

print("✅ Archivos cargados correctamente.")

# ============================================================
# 2. LIMPIEZA DE NOMBRES
# ============================================================

def clean_name(name):
    name = str(name).strip()
    replacements = {
        'Estados Unidos': 'EE. UU.',
        'USA': 'EE. UU.',
        'Bosnia y Herzegovina': 'Bosnia-Herzegovina',
        'Bosnia': 'Bosnia-Herzegovina',
        'República de Corea': 'Corea del Sur',
        'RI de Irán': 'Irán',
        'Islas de Cabo Verde': 'Cabo Verde',
        'Cabo Verde': 'Cabo Verde',
        "Côte d'Ivoire": 'Costa de Marfil',
        'Congo': 'RD Congo',
        'República Democrática del Congo': 'RD Congo',
        'República del Congo': 'Congo',
        'Chequia': 'República Checa',
        'República Checa': 'República Checa',
        'Sudáfrica': 'Sudáfrica',
        'Países Bajos': 'Países Bajos',
        'México': 'México',
        'Catar': 'Catar',
        'España': 'España',
        'Portugal': 'Portugal',
        'Uruguay': 'Uruguay',
        'Costa de Marfil': 'Costa de Marfil',
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name

if 'País' in ranking_fifa.columns:
    ranking_fifa = ranking_fifa.rename(columns={'País': 'Equipo'})

for df in [datos_historicos, datos_mundial, ranking_fifa, transfermarkt]:
    if 'Equipo' in df.columns:
        df['Equipo'] = df['Equipo'].apply(clean_name)
    if 'Equipo_Local' in df.columns:
        df['Equipo_Local'] = df['Equipo_Local'].apply(clean_name)
        df['Equipo_Visitante'] = df['Equipo_Visitante'].apply(clean_name)

cruces_cuartos['Local'] = cruces_cuartos['partido'].apply(lambda x: clean_name(x.split(' vs ')[0]))
cruces_cuartos['Visitante'] = cruces_cuartos['partido'].apply(lambda x: clean_name(x.split(' vs ')[1]))

print("✅ Nombres normalizados.")

# ============================================================
# 3. CONSTRUCCIÓN DEL DATASET DE EQUIPOS
# ============================================================

def get_numeric(df):
    return df[['Equipo'] + df.select_dtypes(include=[np.number]).columns.tolist()]

stats = (get_numeric(ranking_fifa)
         .merge(get_numeric(transfermarkt), on='Equipo', how='left')
         .merge(get_numeric(datos_mundial), on='Equipo', how='left'))
stats = stats.fillna(0)

# Características derivadas
if 'avg_Goles_total' in stats.columns and 'avg_Posesión_total' in stats.columns:
    stats['ratio_goles_posesion'] = stats['avg_Goles_total'] / (stats['avg_Posesión_total'] + 0.01)
if 'avg_Tarjetas_amarillas_total' in stats.columns and 'avg_Faltas_total' in stats.columns:
    stats['ratio_tarjetas'] = stats['avg_Tarjetas_amarillas_total'] / (stats['avg_Faltas_total'] + 0.01)
if 'avg_Goles_total' in stats.columns and 'avg_Remates_a_puerta_total' in stats.columns:
    stats['eficiencia_gol'] = stats['avg_Goles_total'] / (stats['avg_Remates_a_puerta_total'] + 0.01)
if 'avg_Córneres_total' in stats.columns and 'avg_Posesión_total' in stats.columns:
    stats['ratio_corners'] = stats['avg_Córneres_total'] / (stats['avg_Posesión_total'] + 0.01)

equipos_cruces = set(cruces_cuartos['Local']).union(set(cruces_cuartos['Visitante']))
faltantes = [eq for eq in equipos_cruces if eq not in stats['Equipo'].values]
if faltantes:
    print(f"⚠️ Equipos sin estadísticas: {faltantes}. Se añadirán con valores por defecto.")
    for eq in faltantes:
        new_row = {'Equipo': eq}
        for col in stats.columns:
            if col != 'Equipo':
                new_row[col] = 0
        stats = pd.concat([stats, pd.DataFrame([new_row])], ignore_index=True)
    stats = stats.fillna(0)

print("📊 Estadísticas de equipos listas.")

# ============================================================
# 4. ENTRENAMIENTO/CARGA DEL MODELO XGBOOST
# ============================================================

model_path = f"{base_path}/best_model_xgb.pkl"
scaler_path = f"{base_path}/scaler_xgb.pkl"

if os.path.exists(model_path) and os.path.exists(scaler_path):
    print("🔄 Cargando modelo y scaler guardados...")
    best_model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    print("✅ Modelo y scaler cargados correctamente.")
else:
    print("🔄 No se encontraron modelo y scaler guardados. Entrenando desde cero...")

    train_data = []
    for _, row in datos_historicos.iterrows():
        h = row['Equipo_Local']
        a = row['Equipo_Visitante']
        s_h = stats[stats['Equipo'] == h]
        s_a = stats[stats['Equipo'] == a]
        if s_h.empty or s_a.empty:
            continue
        s_h = s_h.iloc[0]
        s_a = s_a.iloc[0]

        match = {}
        for col in stats.select_dtypes(include=[np.number]).columns:
            if col != 'Equipo':
                match[f"{col}_diff"] = s_h[col] - s_a[col]

        if 'ratio_goles_posesion' in stats.columns:
            match['ratio_goles_posesion_diff'] = s_h['ratio_goles_posesion'] - s_a['ratio_goles_posesion']
        if 'ratio_tarjetas' in stats.columns:
            match['ratio_tarjetas_diff'] = s_h['ratio_tarjetas'] - s_a['ratio_tarjetas']
        if 'eficiencia_gol' in stats.columns:
            match['eficiencia_gol_diff'] = s_h['eficiencia_gol'] - s_a['eficiencia_gol']
        if 'ratio_corners' in stats.columns:
            match['ratio_corners_diff'] = s_h['ratio_corners'] - s_a['ratio_corners']

        match['peso_local'] = row.get('Peso_Local', 0.5)

        if row['Goles_Local'] > row['Goles_Visitante']:
            match['target'] = 1
        elif row['Goles_Local'] < row['Goles_Visitante']:
            match['target'] = -1
        else:
            match['target'] = 0

        train_data.append(match)

    df_train = pd.DataFrame(train_data)
    X = df_train.drop(columns=['target'])
    y = df_train['target']
    y = y.replace({-1: 0, 0: 1, 1: 2})

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    param_dist = {
        'n_estimators': [200, 300],
        'max_depth': [6, 8, 10],
        'learning_rate': [0.03, 0.05, 0.07],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }

    model = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss',
                              early_stopping_rounds=10, validation_split=0.1)

    random_search = RandomizedSearchCV(model, param_distributions=param_dist,
                                       n_iter=20, cv=3, scoring='accuracy',
                                       n_jobs=-1, random_state=42, verbose=1)
    random_search.fit(X_scaled, y)

    best_model = random_search.best_estimator_
    print(f"📊 Mejores parámetros: {random_search.best_params_}")
    print(f"📊 Mejor precisión en validación cruzada: {random_search.best_score_:.2%}")

    y_pred_train = best_model.predict(X_scaled)
    train_acc = accuracy_score(y, y_pred_train)
    print(f"📊 Precisión en entrenamiento: {train_acc:.2%}")

    joblib.dump(best_model, model_path)
    joblib.dump(scaler, scaler_path)
    print("✅ Modelo y scaler guardados en Drive.")

print("✅ Modelo XGBoost listo.")

# ============================================================
# 5. FUNCIONES DE PREDICCIÓN Y SIMULACIÓN
# ============================================================

def build_match_features(home, away):
    s_h = stats[stats['Equipo'] == home]
    s_a = stats[stats['Equipo'] == away]
    if s_h.empty or s_a.empty:
        return None
    s_h = s_h.iloc[0]
    s_a = s_a.iloc[0]

    match = {}
    for col in stats.select_dtypes(include=[np.number]).columns:
        if col != 'Equipo':
            match[f"{col}_diff"] = s_h[col] - s_a[col]

    if 'ratio_goles_posesion' in stats.columns:
        match['ratio_goles_posesion_diff'] = s_h['ratio_goles_posesion'] - s_a['ratio_goles_posesion']
    if 'ratio_tarjetas' in stats.columns:
        match['ratio_tarjetas_diff'] = s_h['ratio_tarjetas'] - s_a['ratio_tarjetas']
    if 'eficiencia_gol' in stats.columns:
        match['eficiencia_gol_diff'] = s_h['eficiencia_gol'] - s_a['eficiencia_gol']
    if 'ratio_corners' in stats.columns:
        match['ratio_corners_diff'] = s_h['ratio_corners'] - s_a['ratio_corners']

    match['peso_local'] = 0.55
    return match

def predict_winner(home, away):
    match = build_match_features(home, away)
    if match is None:
        return None

    X_pred = pd.DataFrame([match])
    X_pred_scaled = scaler.transform(X_pred)
    probs = best_model.predict_proba(X_pred_scaled)[0]

    class_mapping = {label: idx for idx, label in enumerate(best_model.classes_)}
    p_home = probs[class_mapping[2]]
    p_draw = probs[class_mapping[1]]
    p_away = probs[class_mapping[0]]

    if p_home >= p_away and p_home >= p_draw:
        ganador = home
    elif p_away >= p_home and p_away >= p_draw:
        ganador = away
    else:
        ganador = home if p_home >= p_away else away

    s_h = stats[stats['Equipo'] == home].iloc[0] if not stats[stats['Equipo'] == home].empty else None
    s_a = stats[stats['Equipo'] == away].iloc[0] if not stats[stats['Equipo'] == away].empty else None
    avg_goals_home = s_h.get('avg_Goles_total', 1.2) if s_h is not None else 1.2
    avg_goals_away = s_a.get('avg_Goles_total', 1.2) if s_a is not None else 1.2
    xg_h = avg_goals_home * (0.8 + 0.4 * p_home)
    xg_a = avg_goals_away * (0.8 + 0.4 * p_away)

    goles_h = int(round(xg_h))
    goles_a = int(round(xg_a))
    if ganador == home and goles_h <= goles_a:
        goles_h = goles_a + 1
    elif ganador == away and goles_a <= goles_h:
        goles_a = goles_h + 1
    if ganador == home and goles_h == goles_a:
        goles_h += 1
    elif ganador == away and goles_h == goles_a:
        goles_a += 1

    goles_h = max(0, goles_h)
    goles_a = max(0, goles_a)

    return ganador, p_home, p_draw, p_away, xg_h, xg_a, goles_h, goles_a

def simular_partido(match_feat, home, away):
    X_pred = pd.DataFrame([match_feat])
    X_pred_scaled = scaler.transform(X_pred)
    probs = best_model.predict_proba(X_pred_scaled)[0]
    class_mapping = {label: idx for idx, label in enumerate(best_model.classes_)}
    p_home = probs[class_mapping[2]]
    p_away = probs[class_mapping[0]]

    s_h = stats[stats['Equipo'] == home].iloc[0] if not stats[stats['Equipo'] == home].empty else None
    s_a = stats[stats['Equipo'] == away].iloc[0] if not stats[stats['Equipo'] == away].empty else None
    avg_goals_home = s_h.get('avg_Goles_total', 1.2) if s_h is not None else 1.2
    avg_goals_away = s_a.get('avg_Goles_total', 1.2) if s_a is not None else 1.2
    lam_home = max(0.1, avg_goals_home + p_home * 1.5)
    lam_away = max(0.1, avg_goals_away + p_away * 1.5)
    g_h = np.random.poisson(lam_home)
    g_a = np.random.poisson(lam_away)

    if g_h > g_a:
        return 'Local', g_h, g_a
    elif g_a > g_h:
        return 'Visitante', g_h, g_a
    else:
        rank_h = s_h.get('Puntuación', 1500) if s_h is not None else 1500
        rank_a = s_a.get('Puntuación', 1500) if s_a is not None else 1500
        prob_local_penal = np.clip(0.5 + (rank_h - rank_a) / 2000.0, 0.2, 0.8)
        if np.random.rand() < prob_local_penal:
            return 'Local', g_h, g_a
        else:
            return 'Visitante', g_h, g_a

# ============================================================
# 6. PREDICCIONES DETERMINISTAS DE CUARTOS
# ============================================================

print("\n" + "="*80)
print("📊 PREDICCIONES DETERMINISTAS - CUARTOS DE FINAL")
print("="*80 + "\n")

partidos_cuartos = []
for _, row in cruces_cuartos.iterrows():
    local = row['Local']
    visitante = row['Visitante']
    feat = build_match_features(local, visitante)
    if feat is not None:
        partidos_cuartos.append((local, visitante, feat))

predicciones = []
clasificados_semifinales = []

for local, visitante, feat in partidos_cuartos:
    result = predict_winner(local, visitante)
    if result is None:
        continue
    ganador, p_h, p_d, p_a, xg_l, xg_v, g_l, g_v = result
    clasificados_semifinales.append(ganador)
    predicciones.append({
        'Local': local,
        'Visitante': visitante,
        'Prob_Local': round(p_h, 3),
        'Prob_Empate': round(p_d, 3),
        'Prob_Visitante': round(p_a, 3),
        'xG_Local': round(xg_l, 2),
        'xG_Visitante': round(xg_v, 2),
        'Goles_Local': g_l,
        'Goles_Visitante': g_v,
        'Ganador': ganador
    })

df_pred = pd.DataFrame(predicciones)

print("📋 TABLA DE PREDICCIONES DETERMINISTAS - CUARTOS")
print(df_pred.to_string(index=False))
print("\n")

print("🎯 CLASIFICADOS A SEMIFINALES (según el modelo determinista)")
for i, eq in enumerate(clasificados_semifinales, 1):
    print(f"{i:2}. {eq}")

# ============================================================
# 7. SIMULACIÓN MONTE CARLO Y SCORE POR PARTIDO
# ============================================================

print("\n🔄 Ejecutando Monte Carlo (1000 iteraciones) para obtener SCORE por partido...")

N_SIM = 1000
# Acumuladores por partido
partido_stats = defaultdict(lambda: {'local_wins': 0, 'draws': 0, 'away_wins': 0,
                                      'goles_local': 0, 'goles_visitante': 0, 'count': 0})

for _ in range(N_SIM):
    for local, visitante, feat in partidos_cuartos:
        resultado, g_l, g_v = simular_partido(feat, local, visitante)
        key = (local, visitante)
        partido_stats[key]['count'] += 1
        partido_stats[key]['goles_local'] += g_l
        partido_stats[key]['goles_visitante'] += g_v
        if resultado == 'Local':
            partido_stats[key]['local_wins'] += 1
        elif resultado == 'Visitante':
            partido_stats[key]['away_wins'] += 1
        else:
            partido_stats[key]['draws'] += 1

# Construir tabla de score por partido
score_partidos = []
for (local, visitante), stats in partido_stats.items():
    total = stats['count']
    score_partidos.append({
        'Local': local,
        'Visitante': visitante,
        'Prob_Local': round(stats['local_wins'] / total, 3),
        'Prob_Empate': round(stats['draws'] / total, 3),
        'Prob_Visitante': round(stats['away_wins'] / total, 3),
        'Goles_Local_Prom': round(stats['goles_local'] / total, 2),
        'Goles_Visitante_Prom': round(stats['goles_visitante'] / total, 2),
        'Partidos_Simulados': total
    })

df_score = pd.DataFrame(score_partidos)
df_score = df_score.sort_values(['Local', 'Visitante'])

print("\n📊 SCORE POR PARTIDO (Monte Carlo) - Promedio de 1000 simulaciones")
print(df_score.to_string(index=False))

# También calculamos las probabilidades de clasificación por equipo
contador_equipos = defaultdict(int)
for _, row in df_pred.iterrows():
    local = row['Local']
    visitante = row['Visitante']
    # Tomamos las probabilidades de clasificación de los partidos individuales
    # (ya están en df_pred, pero usamos las de Monte Carlo para consistencia)
    # Mejor: de las simulaciones, contar cuántas veces ganó cada equipo
    # Ya tenemos los conteos por partido, podemos agregarlos por equipo
    # Pero es más directo usar df_pred que ya tiene el ganador determinista.
    # Sin embargo, para el score, usamos las probabilidades de Monte Carlo.
    pass

# Pero para la tabla de clasificación a semifinales, podemos calcular la probabilidad de cada equipo
# de ganar su partido a partir de df_score
equipo_prob = defaultdict(float)
for _, row in df_score.iterrows():
    local = row['Local']
    visitante = row['Visitante']
    # La probabilidad de que el local avance es Prob_Local + 0.5*Prob_Empate (asumiendo penales 50-50)
    # Pero en la simulación ya se resolvieron los empates con penales, así que Prob_Local ya incluye los penales.
    equipo_prob[local] += row['Prob_Local']
    equipo_prob[visitante] += row['Prob_Visitante']

# Normalizar (cada partido suma 1, pero la suma de probabilidades de todos los equipos es el número de partidos)
# Así que para tener probabilidad de clasificación, la probabilidad de cada equipo es su prob de ganar su partido
df_clas_prob = pd.DataFrame(list(equipo_prob.items()), columns=['Equipo', 'Prob_Clasificacion'])
df_clas_prob = df_clas_prob.sort_values('Prob_Clasificacion', ascending=False)

print("\n📊 PROBABILIDAD DE CLASIFICACIÓN A SEMIFINALES (basada en Monte Carlo)")
print(df_clas_prob.to_string(index=False))

# ============================================================
# 8. RESULTADOS FINALES
# ============================================================

print("\n" + "="*80)
print("RESULTADOS FINALES")
print("="*80)

print("\n📌 CLASIFICADOS A SEMIFINALES (según modelo determinista)")
df_clas = pd.DataFrame({
    'Posición': range(1, len(clasificados_semifinales)+1),
    'Equipo': clasificados_semifinales
})
print(df_clas.to_string(index=False))

print("\n📌 TABLA DE PREDICCIONES DETERMINISTAS")
print(df_pred.to_string(index=False))

print("\n📌 SCORE POR PARTIDO (Monte Carlo)")
print(df_score.to_string(index=False))

print("\n📌 PROBABILIDAD DE CLASIFICACIÓN A SEMIFINALES (Monte Carlo)")
print(df_clas_prob.to_string(index=False))

print("\n✅ Análisis completado.")
print("   - Modelo: XGBoost con ajuste de hiperparámetros")
print("   - Predicciones deterministas con goles enteros")
print("   - Score Monte Carlo por partido: promedios de probabilidades y goles")
print("   - Simulación: 1000 iteraciones")
print("   - Modelo guardado para reutilización")

Mounted at /content/drive
🏆 PREDICCIÓN DE CLASIFICADOS A SEMIFINALES (CUARTOS DE FINAL)

✅ Archivos cargados correctamente.
✅ Nombres normalizados.
📊 Estadísticas de equipos listas.
🔄 Cargando modelo y scaler guardados...
✅ Modelo y scaler cargados correctamente.
✅ Modelo XGBoost listo.

📊 PREDICCIONES DETERMINISTAS - CUARTOS DE FINAL

📋 TABLA DE PREDICCIONES DETERMINISTAS - CUARTOS
    Local  Visitante  Prob_Local  Prob_Empate  Prob_Visitante  xG_Local  xG_Visitante  Goles_Local  Goles_Visitante    Ganador
  Francia  Marruecos       0.819        0.064           0.117      2.38          2.08            3                2    Francia
   España    Bélgica       0.906        0.070           0.024      2.65          1.83            3                2     España
  Noruega Inglaterra       0.204        0.339           0.457      1.96          2.34            2                3 Inglaterra
Argentina      Suiza       0.652        0.257           0.091      1.80          1.41            2        